# 07 비지도 이상탐지 — K-Means vs One-Class SVM

**Phase 7 산출물.** `run_anomaly_detection.py` 스크립트를 노트북으로 전환.

| 항목 | 내용 |
|---|---|
| K-Means 학습 데이터 | `unlabeled_data.csv` 795,315행 (레이블 불필요) |
| OC-SVM 학습 데이터 | `labeled_data` 훈련 fold 양품(Y=0)만 |
| 평가 데이터 | `labeled_data` 5-fold Stratified val fold |
| 평가 지표 | ROC-AUC, PR-AUC (불균형 데이터 적합) |
| 그림 저장 | `results/figures/NB07_fig*.png` |

## Cell 1 — Setup

In [ ]:
import sys, warnings, time
warnings.filterwarnings('ignore')

from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # Run All 안전 모드 — plt.show()는 Jupyter inline으로 표시됨
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import MiniBatchKMeans
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
)

from utils import set_seed, setup_korean_font
from data import load_raw, get_fold
from preprocess import get_feature_cols

set_seed(42)
setup_korean_font()
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR  = PROJECT_ROOT / 'results' / 'tables'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS  = 5
K_VALUES = [5, 10, 20, 50]
print('Setup OK  |  PROJECT_ROOT:', PROJECT_ROOT)

## Cell 2 — 데이터 로드

- `labeled_data`: 지도 학습용 (PassOrFail 포함). **평가 전용**.
- `unlabeled_data`: K-Means 학습용 (795,315행, 레이블 없음).
- 교집합 피처만 사용 (unlabeled에 없는 컬럼 자동 제거).

In [ ]:
df_lab   = load_raw('labeled_data')
df_unlab = load_raw('unlabeled_data')

FEAT_COLS = get_feature_cols(df_lab)
FEAT_COLS = [c for c in FEAT_COLS if c in df_unlab.columns]  # 교집합

X_lab   = df_lab[FEAT_COLS].values
y_lab   = df_lab['PassOrFail'].values
X_unlab = df_unlab[FEAT_COLS].values

print(f'Labeled   : {X_lab.shape}   불량률={y_lab.mean()*100:.2f}%  '
      f'(양품={int((y_lab==0).sum()):,}  불량={int((y_lab==1).sum()):,})')
print(f'Unlabeled : {X_unlab.shape}')
print(f'사용 피처 ({len(FEAT_COLS)}개): {FEAT_COLS}')

## Cell 3 — K-Means 학습 (unlabeled 795K)

795,315행 전체로 `MiniBatchKMeans`를 학습한다.  
`KMeans`보다 ~10× 빠르고 결과는 유사함 (`batch_size=4096, n_init=5`).  
학습 데이터에 레이블이 전혀 필요 없으므로 완전한 **비지도 학습**이다.

**이상 점수** = 테스트 샘플에서 **가장 가까운 centroid까지의 거리**.  
클러스터 중심(정상 패턴)에서 멀수록 이상치로 판단.

In [ ]:
# unlabeled 전체 기준 스케일러 (K-Means에 사용)
scaler_unlab = StandardScaler()
X_unlab_sc   = scaler_unlab.fit_transform(X_unlab)
X_lab_sc     = scaler_unlab.transform(X_lab)   # 이상 점수 계산에 재사용

km_models = {}
print('MiniBatchKMeans 학습 시작...')
for k in K_VALUES:
    t0 = time.time()
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=5,
                         batch_size=4096, max_iter=300)
    km.fit(X_unlab_sc)
    km_models[k] = km
    elapsed = time.time() - t0
    print(f'  k={k:2d}: {elapsed:.1f}s  inertia={km.inertia_:,.0f}')

print('완료.')

## Cell 4 — K-Means 5-fold Val 평가

- val fold: labeled_data의 Stratified 5-fold (data/splits 저장 인덱스 사용)
- 이상 점수: `km.transform(X_va)` → 각 centroid까지 거리 행렬 → `.min(axis=1)`
- k=5, 10, 20, 50 네 가지를 비교해 최적 k를 선택.

In [ ]:
km_fold_rows = []
for fold_i in range(N_FOLDS):
    _, X_va, _, y_va = get_fold(fold_i, X_lab, y_lab)
    X_va_sc = scaler_unlab.transform(X_va)
    for k in K_VALUES:
        dists = km_models[k].transform(X_va_sc)   # (n_va, k)
        score = dists.min(axis=1)                  # 최근접 centroid 거리
        roc = roc_auc_score(y_va, score)
        pr  = average_precision_score(y_va, score)
        km_fold_rows.append({'k': k, 'fold': fold_i, 'roc_auc': roc, 'pr_auc': pr})

km_df = pd.DataFrame(km_fold_rows)
km_summary = (
    km_df.groupby('k')
         .agg(roc_mean=('roc_auc', 'mean'), roc_std=('roc_auc', 'std'),
              pr_mean=('pr_auc',  'mean'), pr_std=('pr_auc',  'std'))
         .reset_index()
)

header = f"{'k':>4}  {'ROC-AUC':>14}  {'PR-AUC':>14}"
print('K-Means 5-fold 결과')
print(header)
print('-' * len(header))
for _, row in km_summary.iterrows():
    print(f"{int(row.k):>4}  {row.roc_mean:.4f}+/-{row.roc_std:.4f}  "
          f"{row.pr_mean:.4f}+/-{row.pr_std:.4f}")

best_k_roc = int(km_summary.loc[km_summary.roc_mean.idxmax(), 'k'])
best_k_pr  = int(km_summary.loc[km_summary.pr_mean.idxmax(),  'k'])
print(f'\nBest ROC-AUC: k={best_k_roc}')
print(f'Best PR-AUC : k={best_k_pr}')

## Cell 5 — One-Class SVM 평가 (labeled 양품만 학습)

- 각 fold의 train fold 중 **양품(y=0)만** 추출해 OC-SVM 학습
- fold별 독립 StandardScaler (leakage 방지)
- `nu=0.02`: 이상치 비율 상한 파라미터 (불량률 0.89% 참고)
- 이상 점수: `-oc.score_samples(X_va)` — 높을수록 이상치

In [ ]:
ocsvm_rows = []
for fold_i in range(N_FOLDS):
    X_tr, X_va, y_tr, y_va = get_fold(fold_i, X_lab, y_lab)
    X_tr_normal = X_tr[y_tr == 0]               # 양품만 추출
    sc = StandardScaler()
    X_tr_n_sc = sc.fit_transform(X_tr_normal)   # 양품 기준으로만 fit
    X_va_sc   = sc.transform(X_va)

    oc = OneClassSVM(kernel='rbf', nu=0.02, gamma='scale')
    oc.fit(X_tr_n_sc)
    score = -oc.score_samples(X_va_sc)          # 높을수록 이상

    roc = roc_auc_score(y_va, score)
    pr  = average_precision_score(y_va, score)
    ocsvm_rows.append({'fold': fold_i, 'roc_auc': roc, 'pr_auc': pr})
    print(f'  fold {fold_i}: ROC={roc:.4f}  PR={pr:.4f}')

ocsvm_df  = pd.DataFrame(ocsvm_rows)
ocsvm_roc = ocsvm_df.roc_auc.mean()
ocsvm_pr  = ocsvm_df.pr_auc.mean()
print(f'\nOC-SVM 평균: ROC={ocsvm_roc:.4f}+/-{ocsvm_df.roc_auc.std():.4f}  '
      f'PR={ocsvm_pr:.4f}+/-{ocsvm_df.pr_auc.std():.4f}')

## Cell 6 — 결과 통합 CSV 저장

`results/tables/anomaly_detection_results.csv` 에 저장.

In [ ]:
summary_rows = []
for _, row in km_summary.iterrows():
    summary_rows.append({
        'method':       f'K-Means (k={int(row.k)})',
        'train_data':   'unlabeled 795K',
        'roc_auc_mean': round(row.roc_mean, 4),
        'roc_auc_std':  round(row.roc_std,  4),
        'pr_auc_mean':  round(row.pr_mean,  4),
        'pr_auc_std':   round(row.pr_std,   4),
    })
summary_rows.append({
    'method':       'One-Class SVM (nu=0.02)',
    'train_data':   'labeled 양품 7,925',
    'roc_auc_mean': round(ocsvm_roc, 4),
    'roc_auc_std':  round(ocsvm_df.roc_auc.std(), 4),
    'pr_auc_mean':  round(ocsvm_pr,  4),
    'pr_auc_std':   round(ocsvm_df.pr_auc.std(),  4),
})

summary_df = pd.DataFrame(summary_rows)
csv_path = TABLES_DIR / 'anomaly_detection_results.csv'
summary_df.to_csv(csv_path, index=False)
print('저장:', csv_path)
print()
print(summary_df.to_string(index=False))

## Cell 7 — Figure 1: K-Means k Ablation

클러스터 수 k = 5, 10, 20, 50에 따른 ROC-AUC / PR-AUC (5-fold 평균 ± 1std) 막대 그래프.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x     = np.arange(len(K_VALUES))
width = 0.6

for ax, metric, ylabel, col in [
    (axes[0], 'roc', 'ROC-AUC', '#4C72B0'),
    (axes[1], 'pr',  'PR-AUC',  '#DD8452'),
]:
    means = km_summary[f'{metric}_mean'].values
    stds  = km_summary[f'{metric}_std'].values
    bars  = ax.bar(x, means, width, color=col, alpha=0.75)
    ax.errorbar(x, means, yerr=stds, fmt='none', color='black', capsize=5)
    ax.set_xticks(x)
    ax.set_xticklabels([f'k={k}' for k in K_VALUES], fontsize=11)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f'K-Means 클러스터 수(k) vs {ylabel}', fontsize=12)
    ax.set_ylim(0, min(1.0, means.max() + 0.15))
    for bar, v in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle(
    'K-Means 거리 기반 이상탐지 — k 튜닝 비교 (unlabeled 학습, labeled 5-fold 평가)',
    fontsize=12)
plt.tight_layout()

fig1_path = FIGURES_DIR / 'NB07_fig1_anomaly_kmeans_k_ablation.png'
plt.savefig(fig1_path, bbox_inches='tight')
plt.show()
print('저장:', fig1_path)

## Cell 8 — Figure 2: ROC & PR 곡선 비교 (Fold 0)

Best k K-Means vs One-Class SVM — Fold 0 기준 ROC 곡선과 PR 곡선.

In [ ]:
# fold 0 데이터
X_tr0, X_va0, y_tr0, y_va0 = get_fold(0, X_lab, y_lab)

# K-Means best k 이상 점수
X_va0_unsc = scaler_unlab.transform(X_va0)
km_score0  = km_models[best_k_roc].transform(X_va0_unsc).min(axis=1)

# OC-SVM 이상 점수 (fold 0 재학습)
sc0        = StandardScaler()
X_tr0_n_sc = sc0.fit_transform(X_tr0[y_tr0 == 0])
X_va0_sc   = sc0.transform(X_va0)
oc0        = OneClassSVM(kernel='rbf', nu=0.02, gamma='scale')
oc0.fit(X_tr0_n_sc)
oc_score0  = -oc0.score_samples(X_va0_sc)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# — ROC 곡선 —
ax = axes[0]
fpr_km, tpr_km, _ = roc_curve(y_va0, km_score0)
fpr_oc, tpr_oc, _ = roc_curve(y_va0, oc_score0)
km_roc0 = roc_auc_score(y_va0, km_score0)
oc_roc0 = roc_auc_score(y_va0, oc_score0)
ax.plot(fpr_km, tpr_km, color='#2196F3', lw=2,
        label=f'K-Means k={best_k_roc}  (AUC={km_roc0:.3f})')
ax.plot(fpr_oc, tpr_oc, color='#E91E63', lw=2,
        label=f'One-Class SVM        (AUC={oc_roc0:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate',  fontsize=11)
ax.set_title('ROC 곡선 (Fold 0)', fontsize=12)
ax.legend(fontsize=9)

# — PR 곡선 —
ax = axes[1]
prec_km, rec_km, _ = precision_recall_curve(y_va0, km_score0)
prec_oc, rec_oc, _ = precision_recall_curve(y_va0, oc_score0)
km_ap0 = average_precision_score(y_va0, km_score0)
oc_ap0 = average_precision_score(y_va0, oc_score0)
ax.plot(rec_km, prec_km, color='#2196F3', lw=2,
        label=f'K-Means k={best_k_roc}  (AP={km_ap0:.3f})')
ax.plot(rec_oc, prec_oc, color='#E91E63', lw=2,
        label=f'One-Class SVM        (AP={oc_ap0:.3f})')
baseline = y_va0.mean()
ax.axhline(baseline, color='gray', ls='--', lw=1,
           label=f'Random (AP={baseline:.3f})')
ax.set_xlabel('Recall',    fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('PR 곡선 (Fold 0)', fontsize=12)
ax.legend(fontsize=9)

plt.suptitle(
    '비지도 이상탐지 비교 — K-Means 거리 vs One-Class SVM (Fold 0)',
    fontsize=12)
plt.tight_layout()

fig2_path = FIGURES_DIR / 'NB07_fig2_anomaly_detection_roc_pr.png'
plt.savefig(fig2_path, bbox_inches='tight')
plt.show()
print('저장:', fig2_path)

## Cell 9 — Figure 3: 이상 점수 분포 (양품 vs 불량)

`labeled_data` 전체를 대상으로 양품/불량의 이상 점수 히스토그램 비교.  
두 분포가 잘 분리될수록 탐지 성능이 높다.

In [ ]:
# labeled 전체 이상 점수 (K-Means)
km_all_score = km_models[best_k_roc].transform(X_lab_sc).min(axis=1)

# labeled 전체 이상 점수 (OC-SVM — 양품 전체로 학습)
sc_all = StandardScaler()
sc_all.fit(X_lab[y_lab == 0])
oc_all = OneClassSVM(kernel='rbf', nu=0.02, gamma='scale')
oc_all.fit(sc_all.transform(X_lab[y_lab == 0]))
oc_all_score = -oc_all.score_samples(sc_all.transform(X_lab))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

titles     = [f'K-Means (k={best_k_roc}) 이상 점수 분포',
              'One-Class SVM 이상 점수 분포']
score_list = [km_all_score, oc_all_score]

for ax, score, title in zip(axes, score_list, titles):
    n_pass = int((y_lab == 0).sum())
    n_fail = int((y_lab == 1).sum())
    ax.hist(score[y_lab == 0], bins=60, alpha=0.65,
            label=f'양품  (n={n_pass:,})', color='#4C72B0', density=True)
    ax.hist(score[y_lab == 1], bins=60, alpha=0.65,
            label=f'불량  (n={n_fail:,})', color='#DD8452', density=True)
    ax.set_xlabel('이상 점수 (높을수록 불량 가능성 높음)', fontsize=10)
    ax.set_ylabel('밀도', fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('양품 vs 불량 이상 점수 분포 비교 (labeled_data 전체)', fontsize=12)
plt.tight_layout()

fig3_path = FIGURES_DIR / 'NB07_fig3_anomaly_score_distribution.png'
plt.savefig(fig3_path, bbox_inches='tight')
plt.show()
print('저장:', fig3_path)

## Cell 10 — 최종 결과 요약

K-Means (best k) vs One-Class SVM의 5-fold 평균 수치 비교.  
가이드북 DNN (ROC-AUC 0.9468 / PR-AUC 0.4655) 대비 비지도 기법 성능 확인.

**해석 포인트:**
- 비지도 방법(레이블 미사용)이 지도학습 DNN 대비 얼마나 차이나는지 정량화
- 이 차이를 **Limitation** 근거로, unlabeled 활용 Semi-supervised Future Work로 연결

In [ ]:
GUIDEBOOK_ROC = 0.9468  # 가이드북 DNN (Phase 6 기준)
GUIDEBOOK_PR  = 0.4655

print('=' * 72)
print('비지도 이상탐지 최종 결과 요약')
print('=' * 72)
print(f"{'방법':<28} {'학습 데이터':<20} {'ROC-AUC':>14} {'PR-AUC':>12}")
print('-' * 72)
for _, row in km_summary.iterrows():
    flag = ' ★ best' if int(row.k) == best_k_roc else ''
    name = f'K-Means (k={int(row.k)})' + flag
    print(f"{name:<28} {'unlabeled 795K':<20} "
          f"{row.roc_mean:.4f}+/-{row.roc_std:.3f}  "
          f"{row.pr_mean:.4f}+/-{row.pr_std:.3f}")
print(f"{'One-Class SVM (nu=0.02)':<28} {'labeled 양품 7,925':<20} "
      f"{ocsvm_roc:.4f}+/-{ocsvm_df.roc_auc.std():.3f}  "
      f"{ocsvm_pr:.4f}+/-{ocsvm_df.pr_auc.std():.3f}")
print('-' * 72)
print(f"{'[참조] 가이드북 DNN (§2.3)':<28} {'labeled 7,996':<20} "
      f"{GUIDEBOOK_ROC:.4f} (ref)          {GUIDEBOOK_PR:.4f} (ref)")
print('=' * 72)
print()
best_km_row = km_summary.loc[km_summary.roc_mean.idxmax()]
gap_roc = GUIDEBOOK_ROC - best_km_row.roc_mean
gap_pr  = GUIDEBOOK_PR  - km_summary.loc[km_summary.pr_mean.idxmax(), 'pr_mean']
print(f'K-Means Best  ROC-AUC: k={best_k_roc}  ({best_km_row.roc_mean:.4f})  '
      f'vs 가이드북 DNN 차이: {gap_roc:+.4f}')
print(f'K-Means Best  PR-AUC : k={best_k_pr}  '
      f'vs 가이드북 DNN 차이: {gap_pr:+.4f}')
print()
print('→ 이 차이가 Semi-supervised Learning (Proposal §3-A) 동기가 됨')